In [1]:
"""
TEC - Otimizador de Portfolio (versao simples e explicada)
============================================================

Esse codigo faz a MESMA coisa que a versao em Julia, mas escrito
de um jeito mais direto: for-loops explicitos, dicionarios ao
inves de matrizes, sem recursao e sem threads.

O que o codigo faz, em portugues simples:
1. Baixa o preco historico de algumas acoes
2. Calcula o retorno diario de cada acao (quanto subiu/desceu, em %)
3. Calcula a media de retorno de cada acao
4. Calcula a covariancia entre cada par de acoes (o "risco conjunto")
5. Testa varias combinacoes de pesos (% investido em cada acao)
6. Escolhe a combinacao com melhor retorno/risco
7. Converte em quantidade de acoes pra comprar
"""

import yfinance as yf

# -----------------------------------------------------------------
# PASSO 1: Baixar os precos historicos
# -----------------------------------------------------------------

acoes = ["PETR4.SA", "VALE3.SA", "ITUB4.SA"]   # comecei com so 3 acoes pra facilitar
data_inicio = "2023-01-01"
data_fim = "2026-06-30"

# Para cada acao, guardamos a lista de precos de fechamento
precos = {}   # exemplo: {"PETR4.SA": [preco_dia1, preco_dia2, ...], ...}

for acao in acoes:
    dados = yf.download(acao, start=data_inicio, end=data_fim)
    # "Close" e o preco de fechamento de cada dia
    precos[acao] = list(dados["Close"])
    print(f"Baixei {len(precos[acao])} dias de preco para {acao}")

# -----------------------------------------------------------------
# PASSO 2: Calcular o retorno diario de cada acao
# -----------------------------------------------------------------
# Retorno do dia = (preco_hoje - preco_ontem) / preco_ontem
# Mede "quanto a acao rendeu" naquele dia, em proporcao (0.01 = 1%)

retornos = {}   # {"PETR4.SA": [retorno_dia1, retorno_dia2, ...], ...}

for acao in acoes:
    lista_precos = precos[acao]
    lista_retornos = []
    for i in range(1, len(lista_precos)):
        preco_ontem = lista_precos[i - 1]
        preco_hoje = lista_precos[i]
        retorno_do_dia = (preco_hoje - preco_ontem) / preco_ontem
        lista_retornos.append(retorno_do_dia)
    retornos[acao] = lista_retornos

# -----------------------------------------------------------------
# PASSO 3: Calcular a media de retorno de cada acao
# -----------------------------------------------------------------
# Media = soma de todos os retornos / quantidade de retornos

medias = {}

for acao in acoes:
    lista = retornos[acao]
    media = sum(lista) / len(lista)
    medias[acao] = media
    print(f"Retorno medio diario de {acao}: {media:.5f}")

# -----------------------------------------------------------------
# PASSO 4: Calcular a covariancia entre cada par de acoes
# -----------------------------------------------------------------
# Covariancia entre A e B mede se elas costumam se mover JUNTAS
# (positiva) ou OPOSTAS (negativa). Formula:
#
#   cov(A,B) = soma( (retorno_A_no_dia - media_A) * (retorno_B_no_dia - media_B) ) / (n - 1)
#
# Quando A == B, isso vira a variancia da propria acao.

def covariancia(acao_a, acao_b):
    lista_a = retornos[acao_a]
    lista_b = retornos[acao_b]
    media_a = medias[acao_a]
    media_b = medias[acao_b]

    n = len(lista_a)
    soma = 0
    for i in range(n):
        soma += (lista_a[i] - media_a) * (lista_b[i] - media_b)

    return soma / (n - 1)

# Guardamos a covariancia de cada par num dicionario.
# Ex: covariancias[("PETR4.SA", "VALE3.SA")] = 0.0001
covariancias = {}
for acao_a in acoes:
    for acao_b in acoes:
        covariancias[(acao_a, acao_b)] = covariancia(acao_a, acao_b)

# -----------------------------------------------------------------
# PASSO 5: Testar varias combinacoes de pesos (forca bruta)
# -----------------------------------------------------------------
# Como temos 3 acoes, um "peso" valido e (w1, w2, w3) onde
# w1 + w2 + w3 = 100, todos >= 0.
#
# Testamos todas as combinacoes em saltos de "passo" (ex: 10 em 10).
# Quanto menor o passo, mais combinacoes -> mais preciso, mais lento.

passo = 10

melhor_resultado = None        # vai guardar a melhor combinacao encontrada
melhor_indice_fo = -999999999  # "nota de qualidade" da melhor carteira ate agora

# Loop duplo: escolhemos o peso da acao 1 e da acao 2.
# O peso da acao 3 e sempre o que sobra, pra garantir soma = 100.
for w1 in range(0, 101, passo):
    for w2 in range(0, 101 - w1, passo):
        w3 = 100 - w1 - w2

        # Convertendo de "0 a 100" para proporcao "0.0 a 1.0"
        pesos = {
            acoes[0]: w1 / 100,
            acoes[1]: w2 / 100,
            acoes[2]: w3 / 100,
        }

        # --- Retorno esperado da carteira ---
        # retorno_carteira = w1*media1 + w2*media2 + w3*media3
        retorno_carteira = 0
        for acao in acoes:
            retorno_carteira += pesos[acao] * medias[acao]

        # --- Risco (variancia) da carteira ---
        # risco_carteira = soma de pesos[i] * pesos[j] * covariancia(i,j)
        # para TODOS os pares (i,j), incluindo quando i == j
        risco_carteira = 0
        for acao_a in acoes:
            for acao_b in acoes:
                risco_carteira += pesos[acao_a] * pesos[acao_b] * covariancias[(acao_a, acao_b)]

        # Evita dividir por zero
        if risco_carteira > 1e-12:
            indice_fo = retorno_carteira / risco_carteira

            # Se essa carteira e melhor que a recordista atual, guarda ela
            if indice_fo > melhor_indice_fo:
                melhor_indice_fo = indice_fo
                melhor_resultado = {
                    "pesos": pesos.copy(),
                    "retorno": retorno_carteira,
                    "risco": risco_carteira,
                }

# -----------------------------------------------------------------
# PASSO 6: Mostrar o resultado
# -----------------------------------------------------------------

print("\n=== MELHOR CARTEIRA ENCONTRADA ===")
for acao in acoes:
    peso_pct = melhor_resultado["pesos"][acao] * 100
    print(f"{acao}: {peso_pct:.1f}%")

print(f"\nRetorno esperado (diario): {melhor_resultado['retorno']:.5f}")
print(f"Risco (variancia): {melhor_resultado['risco']:.8f}")
print(f"Indice retorno/risco: {melhor_indice_fo:.2f}")

# -----------------------------------------------------------------
# PASSO 7 (opcional): converter em quantidade de acoes pra comprar
# -----------------------------------------------------------------

investimento = 10000.0  # quanto voce tem pra investir, em R$

print("\n=== QUANTAS ACOES COMPRAR ===")
for acao in acoes:
    peso = melhor_resultado["pesos"][acao]
    valor_alocado = peso * investimento
    preco_atual = precos[acao][-1]                 # ultimo preco = mais recente
    quantidade = int(valor_alocado // preco_atual)  # // arredonda pra baixo
    print(f"{acao}: comprar {quantidade} acoes (~R$ {valor_alocado:.2f})")

[*********************100%***********************]  1 of 1 completed


Baixei 1 dias de preco para PETR4.SA


[*********************100%***********************]  1 of 1 completed


Baixei 1 dias de preco para VALE3.SA


[*********************100%***********************]  1 of 1 completed

Baixei 1 dias de preco para ITUB4.SA


ZeroDivisionError: division by zero